In [1]:
import pandas as pd
import os

### Reading the data in and creating dataframes

In [ ]:
# read all data in the data folder; will take ~4 minutes to run 
path = "data/"

xl = pd.ExcelFile(os.path.join(path, "2015-2025 SPD Calls with Close Codes.xlsx")) 
# for the SPD Outcomes data, which includes an unnessary "KEY" sheet that we will ignore
sheets = [sheet for sheet in xl.sheet_names if sheet != "KEY"]

print("Reading CSV")
print('2015...')
eug_cad2015 = pd.read_csv(os.path.join(path, "EugeneCAD2015noloc.csv"))
print('2016...')
eug_cad2016 = pd.read_csv(os.path.join(path, "EugeneCAD2016noloc.csv"))
print('2017...')
eug_cad2017 = pd.read_csv(os.path.join(path, "EugeneCAD2017noloc.csv"))
print('2018...')
eug_cad2018 = pd.read_csv(os.path.join(path, "EugeneCAD2018noloc.csv"))
print('2019...')
eug_cad2019 = pd.read_csv(os.path.join(path, "EugeneCAD2019noloc.csv"))
print('2020...')
eug_cad2020 = pd.read_csv(os.path.join(path, "EugeneCAD2020noloc.csv"))
print('2021...')
eug_cad2021 = pd.read_csv(os.path.join(path, "EugeneCAD2021noloc.csv"), low_memory=False)
print('2022...')
eug_cad2022 = pd.read_csv(os.path.join(path, "EugeneCAD2022noloc.csv"))
print('2023...')
eug_cad2023 = pd.read_csv(os.path.join(path, "EugeneCAD2023noloc.csv"))
print('2024...')
eug_cad2024 = pd.read_csv(os.path.join(path, "EugeneCAD2024noloc.csv"))
print('2025...')
eug_cad2025 = pd.read_csv(os.path.join(path, "EugeneCAD2025noloc.csv"), low_memory=False)

print("Reading Excel")
print('SPD Calls for Service...')
spd_calls = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls for Service.xlsx'), sheet_name=None), ignore_index=True)
print("SPD Outcomes...")
spd_outcomes = pd.concat(pd.read_excel(xl, sheet_name=sheets), ignore_index=True)
print('SPD Responding Units...')
spd_units = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Responding Units.xlsx'), sheet_name=None), ignore_index=True)
print("SPD Outcomes...")
spd_outcomes = pd.concat(pd.read_excel(os.path.join(path, '2015-2025 SPD Calls with Close Codes.xlsx'), sheet_name=sheets), ignore_index=True)
print("MCSLC...")
mcslc = pd.concat(pd.read_excel(os.path.join(path, 'MCSLC.xlsx'), sheet_name=None), ignore_index=True)
print("Done")

Reading CSV
2015...
2016...
2017...
2018...
2019...
2020...
2021...
2022...
2023...
2024...
2025...
Reading Excel
SPD Calls for Service...
SPD Outcomes...
SPD Responding Units...
SPD Outcomes...


FileNotFoundError: [Errno 2] No such file or directory: 'data/2015 - 2025 SPD Calls with Close Codes.xlsx'

In [ ]:
dfs = [
    eug_cad2015, eug_cad2016, eug_cad2017, eug_cad2018,
    eug_cad2019, eug_cad2020, eug_cad2021, eug_cad2022,
    eug_cad2023, eug_cad2024, eug_cad2025
]

base_cols = set(dfs[0].columns)

for i, df in enumerate(dfs):
    cols = set(df.columns)
    missing = base_cols - cols
    extra = cols - base_cols
    
    if missing or extra:
        print(f"DataFrame {2015 + i}:")
        if missing:
            print("  Missing:", missing)
        if extra:
            print("  Extra:", extra)

# drop that extra column
eug_cad2025 = eug_cad2025.drop(columns={"month"})

In [ ]:
eug_cad = eug_cad2015
for df in dfs[1:]:
    eug_cad = pd.concat([eug_cad, df], ignore_index=True)


eug_cad.columns

eug_cad.head()

print(eug_cad.shape)

### Cleaning Eugene CAD Dataset

In [ ]:
eug_cad

#### 1. Subset to the columns I need

In [ ]:
columns = ["yr", "calltime", "nature", "closed_as", "primeunit"]
eug = eug_cad[columns].copy()

eug

#### 2. Turn ```calltime``` into a datetime object

In [ ]:
eug["calltime"] = pd.to_datetime(eug["calltime"])
eug.drop('yr', axis=1, inplace=True)
eug.info()

#### 3. Find CAHOOTS calls in Eugene

In [ ]:
date_of_defunding = pd.to_datetime("2025-04-07")

eug_c = eug[(eug['primeunit'].str.contains('J', na=False)) & (eug['calltime'] < date_of_defunding)].copy()
eug_c

#### 4. Find calls in Eugene after CAHOOTS was de-funded

In [ ]:
eug_no_c = eug[(eug["calltime"] > date_of_defunding) & (~eug["primeunit"].str.contains('J', na=False))].copy()
eug_no_c["primeunit"] = eug_no_c["primeunit"].fillna("Unknown")

#### 5. Subset both datasets further to only have welfare check calls

In [ ]:
eug_c = eug_c[eug_c["nature"] == "CHECK WELFARE"]
eug_c

In [ ]:
eug_no_c[eug_no_c["nature"] == "CHECK WELFARE"]

### Cleaning Springfield Call Dataset

In [ ]:
spd_calls

#### 1. Join SPD Calls and Outcome sets

#### 2. Subset to columns I need and rename

In [ ]:
columns = ["Final Call Type", "Call Creation Time", "Primary Responding Unit"]
spd = spd_calls[columns].copy()

spd.rename(columns={
    "Final Call Type": "nature",
    "Call Creation Time": "calltime",
    "Primary Responding Unit": "primeunit"
}, inplace=True)

spd

#### 3. Turn ```calltime``` into a datetime object

In [ ]:
spd["calltime"] = pd.to_datetime(spd["calltime"])
spd.info()

#### 4. Clean the ```primeunit``` column

In [ ]:
spd["primeunit"] = spd["primeunit"].str.strip()

### CSV Output

In [ ]:
eug_no_c.to_csv("cleaned_eug_no_c.csv", index=False)
eug_c.to_csv("cleaned_eug_c.csv", index=False)
spd.to_csv("cleaned_spd.csv", index=False)